In [ ]:
"""
# PhishGuard AI - Train and Compare Model

## Objective

This notebook trains and compares two machine learning classifiers:

1. Decision Tree Classifier
2. Random Forest Classifier

The models are evaluated using:

- Accuracy
- Precision
- Recall
- F1-score

The model with the highest phishing-class F1 score will be selected as the deployment model.

## Label Convention

- 0 = Legitimate
- 1 = Phishing
"""

In [ ]:

import json
import os
import string
import sys

import joblib
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
)
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier


In [ ]:
sys.path.insert(0, os.path.abspath("../backend"))
from app.features import URL_ONLY_FEATURES, compute_features  # noqa: E402


In [ ]:
RANDOM_STATE = 42
CSV = "../PhiUSIIL_Phishing_URL_Dataset.csv"
OUT = "../outputs"
os.makedirs(OUT, exist_ok=True)

In [ ]:
if not os.path.exists(CSV):
    raise SystemExit(f"Dataset not found: {CSV} (place PhiUSIIL CSV in repo root)")

df = pd.read_csv(CSV, encoding="utf-8").drop_duplicates().reset_index(drop=True)
df = df.dropna(subset=["URL"]).reset_index(drop=True)
print(f"Loaded {len(df):,} unique rows from {CSV}")

In [ ]:
y = 1 - df["label"].astype(int)
print(f"Class balance after flip -> Legit(0): {int((y==0).sum()):,} | "
      f"Phish(1): {int((y==1).sum()):,}")

In [ ]:
print("\nBuilding TLD lookup...")
tld_lookup = (
    df.assign(_legit=(y == 0).astype(int))
      .groupby(df["URL"].apply(
          lambda u: u.split("//")[-1].split("/")[0].split(":")[0].rsplit(".", 1)[-1].lower()
          if "." in u.split("//")[-1].split("/")[0] else ""
      ))["_legit"]
      .mean()
      .to_dict()
)
print(f"  {len(tld_lookup)} distinct TLDs")

# Character -> P(char | legit URL). Built from legitimate URLs only, with
# +1 smoothing so any character seen at inference has a non-zero prob.
print("\nBuilding character-frequency lookup from legit URLs...")
counter = {c: 1 for c in string.printable}
total = sum(counter.values())
legit_urls = df.loc[y == 0, "URL"].astype(str)
for u in legit_urls.sample(min(50_000, len(legit_urls)), random_state=RANDOM_STATE):
    for c in u.lower():
        counter[c] = counter.get(c, 0) + 1
        total += 1
char_lookup = {c: v / total for c, v in counter.items()}
print(f"  {len(char_lookup)} distinct characters, total mass = "
      f"{sum(char_lookup.values()):.4f}")

# Save the lookups so features.py can load them at inference
joblib.dump(tld_lookup, os.path.join(OUT, "tld_probability_lookup.joblib"))
joblib.dump(char_lookup, os.path.join(OUT, "char_probability_lookup.joblib"))

In [ ]:
print(f"\nExtracting features from {len(df):,} URLs using compute_features()...")
feats = [compute_features(u, tld_lookup, char_lookup) for u in df["URL"].astype(str)]
X = pd.DataFrame(feats)[URL_ONLY_FEATURES].astype(float)
print(f"  Feature matrix shape: {X.shape}")

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y,
)
print(f"\nTrain: {len(X_train):,} | Test: {len(X_test):,}")


def score(model, name):
    """Fit, evaluate on the held-out test set, return a metrics dict."""
    model.fit(X_train, y_train)
    pred = model.predict(X_test)
    proba = model.predict_proba(X_test)[:, 1]  # P(phish)
    m = {
        "name": name,
        "accuracy": float(accuracy_score(y_test, pred)),
        "precision": float(precision_score(y_test, pred, pos_label=1)),
        "recall": float(recall_score(y_test, pred, pos_label=1)),
        "f1_score": float(f1_score(y_test, pred, pos_label=1)),
        "confusion_matrix": confusion_matrix(y_test, pred).tolist(),
        "n_train": int(len(X_train)),
        "n_test": int(len(X_test)),
        "n_features": len(URL_ONLY_FEATURES),
    }
    print(f"\n{name}")
    print(classification_report(
        y_test, pred, target_names=["Legitimate(0)", "Phishing(1)"], digits=4,
    ))
    return model, m



In [ ]:
dt = DecisionTreeClassifier(
    max_depth=12, min_samples_leaf=5, random_state=RANDOM_STATE,
)
rf = RandomForestClassifier(
    n_estimators=200, max_depth=20, min_samples_leaf=2,
    n_jobs=-1, random_state=RANDOM_STATE,
)

dt_model, dt_metrics = score(dt, "DecisionTreeClassifier(max_depth=12, min_samples_leaf=5)")
rf_model, rf_metrics = score(rf, "RandomForestClassifier(n_estimators=200, max_depth=20)")

# Persist both so the Admin Model Analytics page can compare them
joblib.dump(dt_model, os.path.join(OUT, "dt_model.joblib"))
joblib.dump(rf_model, os.path.join(OUT, "rf_model.joblib"))

# Winner (by F1 on the phishing class) becomes the deployment model
winner_name, winner_model = (
    ("RandomForest", rf_model)
    if rf_metrics["f1_score"] >= dt_metrics["f1_score"]
    else ("DecisionTree", dt_model)
)
print(f"\n>>> Winner (higher phishing-class F1): {winner_name}")

joblib.dump(winner_model, os.path.join(OUT, "deployment_model.joblib"))
joblib.dump(URL_ONLY_FEATURES, os.path.join(OUT, "deployment_features.joblib"))